# Social-Oracle — a quantitative teardown 🔬
### Mention event study · abnormal-return CARs · random-day null · momentum control · the fade · clustering bootstrap · name jackknife · micro-cap capacity

![Signal: None](https://img.shields.io/badge/Signal-None-c0392b?style=flat-square)
![Tradability: Mirage](https://img.shields.io/badge/Tradability-Mirage-c0392b?style=flat-square)
![Pump--and--fade: Directional only](https://img.shields.io/badge/Pump--and--fade-Directional_only-8b949e?style=flat-square)

The deep companion to the [notebook for the curious](01_for_the_curious.ipynb) — *same seven beats, every claim now carrying its standard error.* We take the social-trading claim seriously, then ask whether a public mention is anything more than a **small, late attention bump** once you net out a random day, the momentum the name already had, and realistic micro-cap costs.

> ⚠️ **Not investment advice.** The verdict rests on the fingerprinted run on **1,468** r/WallStreetBets surges (CC-BY `youyanggu/yolostocks-data`, as-of 2026-06-01, fp `1a11c294eeba`) in [`docs/results_wsb.md`](../docs/results_wsb.md): abnormal-return event study with a random-day null, a momentum control, a calendar-block clustering bootstrap and a name jackknife — references in [`docs/references.md`](../docs/references.md).
>
> 💡 **The `💡 In plain words` notes** translate each result back into intuition — so this notebook still reads even if you skim the maths. House style in [METHODOLOGY.md](../../../METHODOLOGY.md).

> 🧪 **What executes vs what's real.** The code cells below run the **synthetic control** — a seeded toy universe with a pump-and-fade deliberately wired in, so the machinery is proven offline and deterministically. The **real numbers** — the fingerprinted run on **1,468** r/WallStreetBets surges (as-of 2026-06-01, fp `1a11c294eeba`) — live in **[docs/results_wsb.md](../docs/results_wsb.md)** and are quoted, sourced, wherever a verdict is called. Don't read a synthetic cell output as the market.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))  # study root (social_oracle/ lives there)
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (9.5, 5.2)
import numpy as np, pandas as pd
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
from social_oracle import data, mentions, eventstudy, benchmark, backtest, robustness

# SYNTHETIC CONTROL — the cells below run a seeded toy universe with a pump-and-fade
# wired in, to prove the machinery offline and deterministically. The REAL numbers
# (1,468 r/WallStreetBets surges) live in ../docs/results_wsb.md, built by
# examples/verify_wsb.py — that run, not these cells, is the study's verdict.
panel, feed = data.synthetic_panel(seed=0)
events, coverage = mentions.to_events(feed, panel)
print(f"SYNTHETIC control: {len(panel)} names, {len(feed)} mentions -> {len(events)} clean events")
print("coverage:", coverage)


SYNTHETIC control: 40 names, 240 mentions -> 232 clean events
coverage: {'raw': 240, 'after_debounce': 232, 'after_cohort': 232, 'no_price': 0, 'too_close_to_edge': 0, 'events': 232}


## Beat 0 · Verdict (real tape)

From [docs/results_wsb.md](../docs/results_wsb.md) — 1,468 events, 224 names, 2021-01-11 → 2025-12-29:

| Axis | Stamp | The decisive numbers |
|---|---|---|
| **Signal** | 🔴 `NONE` | Excess vs the random-day null +0.08% / +0.05% / -0.66% at h = 1/5/21 (p_greater = 0.23/0.40/0.94); clustered bootstrap CIs straddle zero at every horizon. |
| **Tradability** | 🔴 `MIRAGE` | Gross +0.72%/trade is beta (+0.05% abnormal); net hits zero at a 25 bps half-spread; median trade -1.3%; sleeve Sharpe -0.006, max DD -84%. |
| **Pump-and-fade?** | ⚪ `DIRECTIONAL ONLY` | Month-ahead excess -0.66%, up-share 45.7% vs 51.4% random, -1.06% vs already-hot — all pointing down, none significant (p_excess≤0 = 0.85 at h=21). |


## 1 · The claim, as testable hypotheses

H₁: E[CAR_{0→h} | mention] > 0 on **abnormal** returns (name − market), with clustering-robust t > 2, h ∈ {1,5,21}.
H₁′ (the sharp one): that excess survives a **momentum control** and realistic **micro-cap costs**.
H₀: forward abnormal return ≈ 0, or fully explained by prior momentum.

In [2]:
es = eventstudy.event_study(panel, events, horizon=21, pre=5)
es['summary'].loc[[-5,-1,0,1,5,10,21]]

,mean,median,std,pct_positive,tstat,n
rel_day,,,,,,
-5,-0.0581,-0.0605,0.0659,0.2155,-13.4300,232
-1,-0.0145,-0.0119,0.0279,0.3276,-7.9198,232
0,0.0000,0.0000,0.0000,0.0000,NaN,232
1,-0.0026,-0.0033,0.0294,0.4569,-1.3688,232
5,-0.0139,-0.0137,0.0652,0.4310,-3.2507,232
10,-0.0251,-0.0180,0.0894,0.4224,-4.2727,232
21,-0.0258,-0.0197,0.1321,0.4310,-2.9732,232


## 3–4 · The random-day null

`p_greater` = P(a random basket of the same size, drawn from every (name, day) in the universe, beats the mention basket). Small ⇒ the mention adds something.

In [3]:
benchmark.conditional_vs_unconditional(panel, events, horizons=(1,5,21), n_iter=2000)

,n_events,mean_cond,mean_uncond,excess,pct_pos_cond,pct_pos_uncond,p_greater,p_two_sided
horizon,,,,,,,,
1,232,-0.0026,-0.0000,-0.0026,0.4569,0.4991,0.9155,0.1725
5,232,-0.0139,-0.0000,-0.0139,0.4310,0.4997,1.0000,0.0000
21,232,-0.0258,-0.0000,-0.0258,0.4310,0.4969,0.9985,0.0050


## 4 · The momentum control — does 'mentioned' beat 'already hot'?

The confound that makes this study hard: attention follows performance, so a mention rides an existing run. We pit mentions against hot-streak events (a name in its top-decile trailing return) on the same forward abnormal returns. **A mention that clears the random-day null but not this is a momentum sensor.**

In [4]:
hot = mentions.hot_streak_events(panel)
benchmark.excess_vs_alternative(panel, events, hot, horizons=(1,5,21), n_iter=2000)

,n_mention,n_alt,mean_mention,mean_alt,gap,p_mention_gt_alt
horizon,,,,,,
1,232,769,-0.0026,0.0001,-0.0027,0.8835
5,232,769,-0.0139,-0.0015,-0.0124,0.9920
21,232,769,-0.0258,-0.0020,-0.0238,0.9905


## 4 · The fade, clustering, and concentration

**(a) The fade** — mean abnormal CAR by horizon; a peak that reverses is the tell.

In [5]:
robustness.fade_curve(panel, events)

,mean_car,pct_positive,tstat,n
horizon,,,,
1,-0.0026,0.4569,-1.3688,232
2,-0.0060,0.4397,-2.3723,232
3,-0.0092,0.4440,-2.7796,232
5,-0.0139,0.4310,-3.2507,232
10,-0.0251,0.4224,-4.2727,232
21,-0.0258,0.4310,-2.9732,232


**(b) Clustering** — mentions arrive in hype waves; a meme week is one bet, not thirty. A calendar-block bootstrap gives the honest CI on the excess.

In [6]:
{k: round(v,4) for k,v in robustness.block_bootstrap_excess(panel, events, horizon=5, n_iter=2000).items()}

{'mean': -0.0139,
 'ci_low': -0.0223,
 'ci_high': -0.006,
 'p_excess_le_0': 0.9995}

**(c) Concentration** — drop the most-mentioned names one at a time. If the excess collapses, you found a stock, not a skill.

In [7]:
robustness.name_jackknife(panel, events, horizon=5, top=3)

,n_events,mean_cond
dropped,,
(none),232,-0.0139
SYM38,221,-0.0127
SYM36,222,-0.0150
SYM34,223,-0.0113


## 5 · The verdict, with the numbers

The decisive cells are the random-day `p_greater`, the momentum `gap` and its p-value, the block-bootstrap `p_excess_le_0`, the fade-curve peak-vs-month and the jackknife swing. On the **real tape** ([docs/results_wsb.md](../docs/results_wsb.md)) they read:

- random-day null: excess +0.08% / +0.05% / -0.66% at h = 1/5/21, p_greater = 0.23/0.40/0.94 — never significant;
- momentum control: -1.06% vs an already-hot name by a month (p = 0.97);
- clustered bootstrap at h=21: mean -0.66%, CI [-1.94%, +0.62%], p_excess≤0 = 0.85 — the fade is a direction, not a finding;
- jackknife: flat — no single name is carrying (or hiding) anything.

**Signal `NONE`** — the bump never clears either null — and **Tradability `MIRAGE` regardless** once beat 6 charges costs. The pump-and-fade shape the synthetic cells above display so cleanly is, on the real tape, only **directional**.

## 6 · Could you trade it — costs and capacity

Enter at the next open (you saw the tweet when everyone did), hold a fixed window, charge a micro-cap spread twice. Then ask how much size the names can even absorb before your own order is the move.

In [8]:
res = backtest.run(panel, events, hold_days=10)
print({k:(round(v,4) if isinstance(v,float) else v) for k,v in res.stats.items()})
print('\ncost sweep (half-spread bps -> mean net trade):')
display(backtest.cost_sweep(panel, events))
print('capacity at a nominal 50bp edge:')
backtest.capacity(panel, events, edge_bps=50.0)

{'n_trades': 232, 'mean_gross': -0.0179, 'mean_abnormal': -0.0252, 'mean_net': -0.0251, 'median_net': -0.0222, 'win_rate_net': 0.4095, 'tstat_net': -4.3083, 'sleeve_sharpe': -3.0105, 'sleeve_max_drawdown': -0.7837, 'total_return': -0.7683}

cost sweep (half-spread bps -> mean net trade):


,round_trip_bps,mean_net,win_rate_net,n_trades
half_spread_bps,,,,
5,32.0000,-0.0211,0.4310,232
15,52.0000,-0.0231,0.4224,232
25,72.0000,-0.0251,0.4095,232
50,122.0000,-0.0301,0.3836,232
100,222.0000,-0.0401,0.3276,232


capacity at a nominal 50bp edge:


{'median_adv_usd': 1509846.7366831321,
 'capacity_usd_per_trade': 3774.616841707831,
 'edge_bps': 50.0}

**The data-mining check.** Hold period, lookback, cooldown — try enough knobs and one cell shines. Deflate the best Sharpe for the number of configs tried.

In [9]:
import itertools
rows = []
for hold in (3,5,10,21):
    r = backtest.run(panel, events, hold_days=hold)
    rows.append({'hold': hold, 'sharpe': r.stats.get('sleeve_sharpe', float('nan')),
                 'n': r.stats['n_trades']})
scan = pd.DataFrame(rows).sort_values('sharpe', ascending=False)
best = scan.iloc[0]
dsr = robustness.deflated_sharpe(best.sharpe, n_trials=len(scan), n_obs=int(best.n))
print(f"best hold={int(best.hold)}d Sharpe={best.sharpe:.2f}; deflated over {len(scan)} configs: {dsr:.3f}")
scan

best hold=21d Sharpe=-1.47; deflated over 4 configs: 0.007


,hold,sharpe,n
3,21,-1.4727,232
2,10,-3.0105,232
1,5,-3.2336,232
0,3,-3.2370,232


## 7 · Going further

- **Short the fade** — the month-ahead drift points down but isn't significant ([docs/results_wsb.md](../docs/results_wsb.md)); test the inverted trade directly, net of micro-cap borrow, before believing it.
- **Beta-estimated abnormal return** to replace the β=1 market adjustment.
- **Conviction / first-mention / pile-on** splits of the feed.
- **A survivorship-clean feed** — the real run drops 42 delisted names for lack of a price history; recovering them would make the month-ahead numbers *worse*, not better, and might turn the directional fade into a finding.

Engine: [`../../../quantlab/`](../../../quantlab/). Method: [`METHODOLOGY.md`](../../../METHODOLOGY.md).